# Numerical Values of Angular-Momentum Coefficients

Exact numerical values for the three coefficients, in the form the book's
numerical tables use — a **signed square root of a rational fraction**, plus a
decimal:

$$\langle 1\,0,\;1\,0 \mid 2\,0\rangle \;=\; \left(\tfrac{2}{3}\right)^{1/2}
\;=\; 0.8164965809$$

| Section | Book tables | Object |
|---|---|---|
| 8.13 | 8.11 | Clebsch–Gordan $\langle a\alpha, b\beta \mid c\gamma\rangle$ |
| 9.12 | 9.9–9.11 | $6j$ $\begin{Bmatrix}a&b&c\\d&e&f\end{Bmatrix}$ |
| 10.12 | 10.13–10.14 | $9j$ (all nine arguments) |

Everything is computed in exact rational arithmetic (SymPy's `CG`,
`wigner_6j`, `wigner_9j`), so the fraction under the root is exact — the
decimal is derived from it, never the other way round.

Companion to `vmk_algebraic_tables.ipynb`, which generates the *algebraic*
formulas. The last section cross-checks the two against each other.

## Setup

In [ ]:
import sys, pathlib
from fractions import Fraction

import sympy as sp
from sympy import Rational, nsimplify
from sympy.physics.quantum.cg import CG
from sympy.physics.wigner import wigner_6j, wigner_9j
from IPython.display import display, Math, HTML

here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / "scripts" / "gen_8_12_cg_tables.py").exists():
        ROOT = cand
        sys.path.insert(0, str(cand / "scripts"))
        break
else:
    ROOT = here          # the algebraic cross-check section needs this; the rest does not

try:
    import ipywidgets as W
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

DIGITS = 10          # significant digits in the decimal form

print(f"repo root : {ROOT}")
print(f"widgets   : {'yes' if HAS_WIDGETS else 'no  (edit-and-rerun cells still work)'}")

## Helpers

In [ ]:
# Output spacing, as in the algebraic notebook. Keep `gap` >= 0: a negative
# margin makes tall results collide with the text underneath.
STYLE = {"indent": "1.7em", "gap": "0.55em", "below": "1.1em", "lead": "1.6"}


def _R(x):
    """Anything (0.5, '1/2', Rational) -> exact Rational."""
    if isinstance(x, float):
        return Rational(Fraction(x).limit_denominator(4096))
    return Rational(x)


def _fmt(x):
    return str(_R(x))


def root_form(v):
    """(sign, r) with  v == sign*sqrt(r)  and r a non-negative Rational.

    Returns None if v is not of that form (which should not happen for these
    coefficients -- if it does, the caller falls back to the raw exact value).
    """
    v = sp.simplify(nsimplify(v))
    if v == 0:
        return 0, sp.Integer(0)
    sq = sp.simplify(v ** 2)
    if not sq.is_Rational:
        return None
    return (-1 if v.is_negative else 1), Rational(sq)


def root_latex(v):
    """LaTeX for the signed-root form:  -\left(\frac{3}{10}\right)^{1/2}."""
    rf = root_form(v)
    if rf is None:
        return sp.latex(v)
    sign, r = rf
    if r == 0:
        return "0"
    rt = sp.sqrt(r)
    body = sp.latex(rt) if rt.is_Rational else rf"\left({sp.latex(r)}\right)^{{1/2}}"
    return ("-" if sign < 0 else "") + body


def decimal(v, digits=None):
    return sp.N(v, digits or DIGITS)


def _bundle(obj):
    """An IPython display object -> an output-widget mime bundle."""
    data = {}
    if hasattr(obj, "_repr_latex_"):
        data["text/latex"] = obj._repr_latex_()
    if hasattr(obj, "_repr_html_"):
        data["text/html"] = obj._repr_html_()
    data.setdefault("text/plain", "")
    return {"output_type": "display_data", "data": data, "metadata": {}}


def publish(objs, into=None):
    """Show `objs` in the cell, or place them in an Output widget.

    Assigning `into.outputs` is deliberate: capturing with `with out: display(...)`
    is replayed by some front ends (VS Code), which makes every line appear
    twice. Direct assignment puts exactly one copy in the widget.
    """
    if into is None:
        for o in objs:
            display(o)
    else:
        into.outputs = tuple(_bundle(o) for o in objs)


def _emit(symbol, v, note=None, into=None):
    """symbol = (root of fraction) = decimal, indented; or a note if it vanishes."""
    tail = (f"margin:{STYLE['gap']} 0 {STYLE['below']} {STYLE['indent']};"
            f"font-size:90%;line-height:{STYLE['lead']}")
    if note:
        publish([HTML(f"<div style='{tail};color:#a33'>{note}</div>")], into)
        return
    publish([Math(r"\hspace{1.5em}" + symbol + r"\;=\;" + root_latex(v)
                  + r"\;=\;" + sp.latex(decimal(v))),
             HTML(f"<div style='{tail};color:#666'>exact: <code>{v}</code></div>")],
            into)


def _tri(j1, j2, j3):
    """Triangle condition, including the integer-perimeter requirement."""
    j1, j2, j3 = _R(j1), _R(j2), _R(j3)
    return abs(j1 - j2) <= j3 <= j1 + j2 and (j1 + j2 + j3) % 1 == 0


def _bad_momentum(*js):
    for j in js:
        j = _R(j)
        if j < 0 or (2 * j) % 1:
            return f"{j} is not a non-negative multiple of 1/2"
    return None

`root_form` squares the exact value and checks the result is rational — that
is what makes the $\pm\sqrt{p/q}$ presentation exact rather than a decimal
fitted after the fact.

---
## Clebsch–Gordan coefficients

$$\langle a\,\alpha,\; b\,\beta \mid c\,\gamma\rangle$$

$\gamma$ is fixed by $\gamma=\alpha+\beta$ (the coefficient vanishes otherwise),
so it is derived rather than entered. Values may be integers or half-integers.

In [ ]:
def clebsch(a, alpha, b, beta, c, gamma=None, into=None):
    """Numeric Clebsch-Gordan coefficient. Returns the exact SymPy value."""
    a, alpha, b, beta, c = map(_R, (a, alpha, b, beta, c))
    gamma = alpha + beta if gamma is None else _R(gamma)
    sym = (rf"\langle {_fmt(a)}\,{_fmt(alpha)},\; {_fmt(b)}\,{_fmt(beta)} \mid "
           rf"{_fmt(c)}\,{_fmt(gamma)}\rangle")

    bad = _bad_momentum(a, b, c)
    reasons = [bad] if bad else []
    for j, m, nm in ((a, alpha, "a"), (b, beta, "b"), (c, gamma, "c")):
        if abs(m) > j or (j - m) % 1:
            reasons.append(f"projection {m} invalid for {nm}={j}")
    if gamma != alpha + beta:
        reasons.append(rf"gamma must equal alpha+beta = {_fmt(alpha+beta)}")
    if not reasons and not _tri(a, b, c):
        reasons.append(f"({_fmt(a)}, {_fmt(b)}, {_fmt(c)}) fails the triangle rule")
    if reasons:
        _emit(sym, 0, note="vanishes: " + "; ".join(reasons), into=into)
        return sp.Integer(0)

    try:
        v = sp.simplify(CG(a, alpha, b, beta, c, gamma).doit())
    except ValueError as exc:
        _emit(sym, 0, note=f"vanishes: {exc}", into=into)
        return sp.Integer(0)
    _emit(sym, v, into=into)
    return v

In [ ]:
# ---- edit and re-run ----
A, ALPHA, B, BETA, Cc = 1, 0, 1, 0, 2

clebsch(A, ALPHA, B, BETA, Cc);

In [ ]:
if HAS_WIDGETS:
    def _num(value=0.0, lo=-20.0, hi=20.0, width="70px"):
        return W.BoundedFloatText(value=value, min=lo, max=hi, step=0.5,
                                  layout=W.Layout(width=width))

    def _lab(text, width="2.2em"):
        return W.HTML(f"<div style='text-align:right;font-style:italic;"
                      f"font-family:Georgia,serif;font-size:115%'>{text}</div>",
                      layout=W.Layout(width=width, margin="0 6px 0 0"))

    def _pairs(*pairs, gap="16px"):
        return W.HBox([W.HBox([_lab(t), w],
                              layout=W.Layout(align_items="center",
                                              margin=f"0 {gap} 0 0"))
                       for t, w in pairs],
                      layout=W.Layout(align_items="center", margin="2px 0"))

    def _title(text, tex):
        cap = W.HTML(f"<div style='font-weight:600'>{text}</div>")
        sym = W.Output()
        publish([Math(tex)], sym)          # assign, never capture
        return W.VBox([cap, sym], layout=W.Layout(margin="0 0 4px 0"))

    def _note(out, msg, colour="#888"):
        publish([HTML(f"<div style='margin-left:{STYLE['indent']};color:{colour};"
                      f"font-size:90%'>{msg}</div>")], out)

    def _stale(out):
        # inputs changed: drop the old result rather than let it sit there
        # disagreeing with the controls
        _note(out, "Set the arguments, then press <b>Compute</b>.")

    def _cg_ui():
        a  = _num(1.0, lo=0.0); al = _num(0.0)
        b  = _num(1.0, lo=0.0); be = _num(0.0)
        c  = _num(2.0, lo=0.0)
        # gamma is derived, not entered: shown inline with the inputs, its
        # value upright so it does not read as another italic symbol
        gam = W.HTML(layout=W.Layout(width="70px"))
        out = W.Output(layout=W.Layout(margin="6px 0 4px 0"))

        def recompute(*_):
            gam.value = ("<div style='font-style:normal;color:#333;"
                         "padding:3px 0 3px 2px'>"
                         f"{_fmt(_R(al.value) + _R(be.value))}</div>")
            clebsch(a.value, al.value, b.value, be.value, c.value, into=out)

        for ctl in (a, al, b, be, c):
            ctl.observe(recompute, "value")
        recompute()
        return W.VBox([
            _title("Clebsch&ndash;Gordan coefficient",
                   r"\langle a\,\alpha,\; b\,\beta \mid c\,\gamma\rangle,"
                   r"\qquad \gamma = \alpha+\beta"),
            _pairs(("a", a), ("α", al), ("b", b), ("β", be), ("c", c),
                   ("γ", gam)),
            out])

    display(_cg_ui())
else:
    print("ipywidgets not installed - use the edit-and-rerun cell above.")

In [ ]:
def cg_table(a, b, c, decimals=False):
    """All non-zero coefficients for fixed (a, b, c) -- the shape of Table 8.11."""
    a, b, c = map(_R, (a, b, c))
    if not _tri(a, b, c):
        return HTML(f"<b>({_fmt(a)}, {_fmt(b)}, {_fmt(c)}) fails the triangle rule</b>")
    alphas = [a - i for i in range(int(2 * a) + 1)]
    betas  = [b - i for i in range(int(2 * b) + 1)]
    head = [r"\alpha \backslash \beta"] + [_fmt(x) for x in betas]
    rows = [" & ".join(head) + r" \\ \hline"]
    for al in alphas:
        cells = []
        for be in betas:
            ga = al + be
            if abs(ga) > c:
                cells.append("-")
                continue
            v = sp.simplify(CG(a, al, b, be, c, ga).doit())
            cells.append(sp.latex(decimal(v, 6)) if decimals else root_latex(v))
        rows.append(" & ".join([_fmt(al)] + cells) + r" \\")
    return Math(r"\begin{array}{c|" + "c" * len(betas) + "}"
                + " ".join(rows) + r"\end{array}")


cg_table(1, 1, 2)

---
## $6j$ symbols

$$\begin{Bmatrix} a & b & c \\ d & e & f\end{Bmatrix}$$

The four triads $(a,b,c)$, $(a,e,f)$, $(d,b,f)$, $(d,e,c)$ must each satisfy the
triangle rule; the symbol vanishes otherwise, and the reason is reported.

In [ ]:
SIXJ_TRIADS = ((0, 1, 2), (0, 4, 5), (3, 1, 5), (3, 4, 2))   # index into (a..f)


def sixj(a, b, c, d, e, f, into=None):
    """Numeric 6j symbol. Returns the exact SymPy value."""
    j = list(map(_R, (a, b, c, d, e, f)))
    sym = (rf"\begin{{Bmatrix}} {_fmt(j[0])} & {_fmt(j[1])} & {_fmt(j[2])} \\ "
           rf"{_fmt(j[3])} & {_fmt(j[4])} & {_fmt(j[5])}\end{{Bmatrix}}")

    bad = _bad_momentum(*j)
    reasons = [bad] if bad else []
    names = "abcdef"
    for t in SIXJ_TRIADS:
        if not _tri(*(j[i] for i in t)):
            reasons.append("(" + ", ".join(names[i] for i in t) + ") = ("
                           + ", ".join(_fmt(j[i]) for i in t) + ") fails the triangle rule")
    if reasons:
        _emit(sym, 0, note="vanishes: " + "; ".join(reasons), into=into)
        return sp.Integer(0)

    try:
        v = sp.simplify(wigner_6j(*j, prec=None))
    except ValueError as exc:                 # sympy raises rather than returning 0
        _emit(sym, 0, note=f"vanishes: {exc}", into=into)
        return sp.Integer(0)
    _emit(sym, v, into=into)
    return v

In [ ]:
# ---- edit and re-run ----
J6 = (1, 1, 1, 1, 1, 1)          # a b c d e f

sixj(*J6);

In [ ]:
if HAS_WIDGETS:
    def _6j_ui():
        w = [_num(1.0, lo=0.0) for _ in range(6)]
        out = W.Output(layout=W.Layout(margin="6px 0 4px 0"))

        def recompute(*_):
            sixj(*[x.value for x in w], into=out)

        for ctl in w:
            ctl.observe(recompute, "value")
        recompute()
        return W.VBox([
            _title("$6j$ symbol",
                   r"\begin{Bmatrix} a & b & c \\ d & e & f\end{Bmatrix}"),
            _pairs(("a", w[0]), ("b", w[1]), ("c", w[2])),
            _pairs(("d", w[3]), ("e", w[4]), ("f", w[5])),
            out])

    display(_6j_ui())
else:
    print("ipywidgets not installed - use the edit-and-rerun cell above.")

---
## $9j$ symbols

$$\begin{Bmatrix} a & b & c \\ d & e & f \\ g & h & i\end{Bmatrix}$$

All six triads — three rows and three columns — must satisfy the triangle rule.
The input grid mirrors the symbol.

In [ ]:
NINEJ_TRIADS = ((0, 1, 2), (3, 4, 5), (6, 7, 8),      # rows
                (0, 3, 6), (1, 4, 7), (2, 5, 8))      # columns


def ninej(*js, into=None):
    """Numeric 9j symbol from nine arguments a b c d e f g h i."""
    if len(js) != 9:
        raise ValueError("a 9j symbol needs exactly nine arguments")
    j = list(map(_R, js))
    sym = (r"\begin{Bmatrix} "
           + r" \\ ".join(" & ".join(_fmt(j[3 * r + k]) for k in range(3))
                          for r in range(3))
           + r"\end{Bmatrix}")

    bad = _bad_momentum(*j)
    reasons = [bad] if bad else []
    names = "abcdefghi"
    for t in NINEJ_TRIADS:
        if not _tri(*(j[i] for i in t)):
            reasons.append("(" + ", ".join(names[i] for i in t) + ") fails the triangle rule")
    if reasons:
        _emit(sym, 0, note="vanishes: " + "; ".join(reasons), into=into)
        return sp.Integer(0)

    try:
        v = sp.simplify(wigner_9j(*j, prec=None))
    except ValueError as exc:
        _emit(sym, 0, note=f"vanishes: {exc}", into=into)
        return sp.Integer(0)
    _emit(sym, v, into=into)
    return v

In [ ]:
# ---- edit and re-run ----
J9 = (Rational(1, 2), Rational(1, 2), 1,          # {1/2 1/2 1}
      Rational(1, 2), Rational(1, 2), 1,          # {1/2 1/2 1}  =  1/9
      1,             1,             2)            # { 1   1  2}

ninej(*J9);

In [ ]:
if HAS_WIDGETS:
    def _9j_ui():
        start = [0.5, 0.5, 1.0, 0.5, 0.5, 1.0, 1.0, 1.0, 2.0]   # a non-zero 9j
        w = [_num(v, lo=0.0) for v in start]
        out = W.Output(layout=W.Layout(margin="6px 0 4px 0"))

        def recompute(*_):
            ninej(*[x.value for x in w], into=out)

        for ctl in w:
            ctl.observe(recompute, "value")
        recompute()
        names = "abcdefghi"
        rows = [_pairs(*[(names[3 * r + k], w[3 * r + k]) for k in range(3)])
                for r in range(3)]
        return W.VBox([
            _title("$9j$ symbol",
                   r"\begin{Bmatrix} a & b & c \\ d & e & f \\ g & h & i"
                   r"\end{Bmatrix}"),
            *rows, out])

    display(_9j_ui())
else:
    print("ipywidgets not installed - use the edit-and-rerun cell above.")

---
## $12j$ symbols of the first kind

$$\left\{\begin{array}{llll}
 a_1 & a_2 & a_3 & a_4 \\
 \quad b_{12} & \quad b_{23} & \quad b_{34} & \quad b_{41} \\
 c_1 & c_2 & c_3 & c_4 \end{array}\right\}$$

SymPy has no $12j$, so this uses the book's own single sum over four $6j$
symbols, **eq 10.13.6**:

$$\{12j(\mathrm{I})\} = \sum_x (-1)^{S-x}(2x+1)\,
\begin{Bmatrix}a_1&a_2&b_{12}\\c_2&c_1&x\end{Bmatrix}
\begin{Bmatrix}a_2&a_3&b_{23}\\c_3&c_2&x\end{Bmatrix}
\begin{Bmatrix}a_3&a_4&b_{34}\\c_4&c_3&x\end{Bmatrix}
\begin{Bmatrix}a_4&c_1&b_{41}\\a_1&c_4&x\end{Bmatrix}$$

The implementation is imported from `scripts/check_10_13.py`, where it is the
reference definition every other §10.13 form is validated against — so it is
the same code the checker exercises, not a second copy.

Eight triads and two tetrads must hold; the reason for a vanishing symbol is
reported.

In [ ]:
from check_10_13 import TW as _tw1, TW2 as _tw2, valid12, valid12II

def _tetrad(j1, j2, j3, j4):
    """Quadrilateral condition: integer perimeter, no side exceeding the rest."""
    j1, j2, j3, j4 = map(_R, (j1, j2, j3, j4))
    return ((j1 + j2 + j3 + j4) % 1 == 0
            and j1 <= j2 + j3 + j4 and j2 <= j1 + j3 + j4
            and j3 <= j1 + j2 + j4 and j4 <= j1 + j2 + j3)


def _why(triads, tetrads):
    """Names of the constraints that fail (used only to explain a zero)."""
    out = [f"({nm}) fails the triangle rule" for nm, t in triads if not _tri(*t)]
    out += [f"({nm}) fails the quadrilateral rule" for nm, q in tetrads if not _tetrad(*q)]
    return out


TRIADS_12I = "a1 b12 a2|a2 b23 a3|a3 b34 a4|a4 b41 c1|c1 b12 c2|c2 b23 c3|c3 b34 c4|c4 b41 a1"
TETRADS_12I = "a1 c1 a3 c3|a2 c2 a4 c4"


def twelvej1(a1, a2, a3, a4, b12, b23, b34, b41, c1, c2, c3, c4, into=None):
    """12j(I) via eq 10.13.6. Argument order as in \\twelvejI."""
    v = list(map(_R, (a1, a2, a3, a4, b12, b23, b34, b41, c1, c2, c3, c4)))
    a1, a2, a3, a4, b12, b23, b34, b41, c1, c2, c3, c4 = v
    sym = (r"\left\{\begin{array}{llll} "
           + " & ".join(_fmt(x) for x in (a1, a2, a3, a4)) + r" \\ "
           + " & ".join(r"\quad " + _fmt(x) for x in (b12, b23, b34, b41)) + r" \\ "
           + " & ".join(_fmt(x) for x in (c1, c2, c3, c4))
           + r"\end{array}\right\}")

    bad = _bad_momentum(*v)
    reasons = [bad] if bad else []
    names = TRIADS_12I.split("|")
    vals = [(a1, b12, a2), (a2, b23, a3), (a3, b34, a4), (a4, b41, c1),
            (c1, b12, c2), (c2, b23, c3), (c3, b34, c4), (c4, b41, a1)]
    qn = TETRADS_12I.split("|")
    qv = [(a1, c1, a3, c3), (a2, c2, a4, c4)]
    reasons += _why(list(zip(names, vals)), list(zip(qn, qv)))
    if not reasons and not valid12(*v):
        reasons.append("fails the validity test of check_10_13.valid12")
    if reasons:
        _emit(sym, 0, note="vanishes: " + "; ".join(reasons), into=into)
        return sp.Integer(0)

    val = sp.simplify(_tw1(*v))
    _emit(sym, val, into=into)
    return val

In [ ]:
# ---- edit and re-run ----  (a1 a2 a3 a4 | b12 b23 b34 b41 | c1 c2 c3 c4)
J12I = (1, 1, 1, 1,
        1, 1, 1, 1,
        1, 1, 1, 1)

twelvej1(*J12I);

In [ ]:
if HAS_WIDGETS:
    def _12j1_ui():
        start = [1.0] * 12
        w = [_num(v, lo=0.0) for v in start]
        labs = ["a₁", "a₂", "a₃", "a₄", "b₁₂", "b₂₃", "b₃₄", "b₄₁",
                "c₁", "c₂", "c₃", "c₄"]
        go  = W.Button(description="Compute", button_style="primary",
                       layout=W.Layout(width="110px"))
        out = W.Output(layout=W.Layout(margin="6px 0 4px 0"))

        def recompute(*_):
            _note(out, "computing &hellip;", colour="#666")
            twelvej1(*[x.value for x in w], into=out)

        for ctl in w:
            ctl.observe(lambda *_: _stale(out), "value")
        go.on_click(recompute)
        _stale(out)
        rows = [_pairs(*[(labs[4 * r + k], w[4 * r + k]) for k in range(4)],
                       gap="12px") for r in range(3)]
        return W.VBox([
            _title("$12j$(I) symbol",
                   r"\left\{\begin{array}{llll} a_1 & a_2 & a_3 & a_4 \\"
                   r" \quad b_{12} & \quad b_{23} & \quad b_{34} & \quad b_{41} \\"
                   r" c_1 & c_2 & c_3 & c_4 \end{array}\right\}"),
            *rows,
            W.HBox([go], layout=W.Layout(margin="6px 0 0 0")),
            out])

    display(_12j1_ui())
else:
    print("ipywidgets not installed - use the edit-and-rerun cell above.")

---
## $12j$ symbols of the second kind

$$\left\{\begin{array}{cccc}
 -   & a_2 & a_3 & a_4 \\
 b_1 & -   & b_3 & b_4 \\
 c_1 & c_2 & -   & c_4 \\
 d_1 & d_2 & d_3 & - \end{array}\right\}$$

Again a single sum over four $6j$ symbols, **eq 10.13.26**:

$$\{12j(\mathrm{II})\} = (-1)^{b_3-a_4-d_1+c_2}\sum_x (2x+1)\,
\begin{Bmatrix}a_3&b_4&x\\b_1&d_3&b_3\end{Bmatrix}
\begin{Bmatrix}a_3&b_4&x\\c_4&a_2&a_4\end{Bmatrix}
\begin{Bmatrix}b_1&d_3&x\\d_2&c_1&d_1\end{Bmatrix}
\begin{Bmatrix}c_4&a_2&x\\d_2&c_1&c_2\end{Bmatrix}$$

Argument order follows `\twelvejII`: $(a_2\,a_3\,a_4 \mid b_1\,b_3\,b_4 \mid
c_1\,c_2\,c_4 \mid d_1\,d_2\,d_3)$ — the four rows of the array with the
diagonal dashes omitted. Eight triads and three tetrads must hold.

In [ ]:
TRIADS_12II = ("a2 a3 a4|b1 b3 b4|c1 c2 c4|d1 d2 d3|"
               "b1 c1 d1|a2 c2 d2|a3 b3 d3|a4 b4 c4")
TETRADS_12II = "a2 c4 d3 b1|a3 b4 d2 c1|a4 b3 c2 d1"


def twelvej2(a2, a3, a4, b1, b3, b4, c1, c2, c4, d1, d2, d3, into=None):
    """12j(II) via eq 10.13.26. Argument order as in \\twelvejII."""
    v = list(map(_R, (a2, a3, a4, b1, b3, b4, c1, c2, c4, d1, d2, d3)))
    a2, a3, a4, b1, b3, b4, c1, c2, c4, d1, d2, d3 = v
    dash = "-"
    sym = (r"\left\{\begin{array}{cccc} "
           + " & ".join([dash, _fmt(a2), _fmt(a3), _fmt(a4)]) + r" \\ "
           + " & ".join([_fmt(b1), dash, _fmt(b3), _fmt(b4)]) + r" \\ "
           + " & ".join([_fmt(c1), _fmt(c2), dash, _fmt(c4)]) + r" \\ "
           + " & ".join([_fmt(d1), _fmt(d2), _fmt(d3), dash])
           + r"\end{array}\right\}")

    bad = _bad_momentum(*v)
    reasons = [bad] if bad else []
    vals = [(a2, a3, a4), (b1, b3, b4), (c1, c2, c4), (d1, d2, d3),
            (b1, c1, d1), (a2, c2, d2), (a3, b3, d3), (a4, b4, c4)]
    qv = [(a2, c4, d3, b1), (a3, b4, d2, c1), (a4, b3, c2, d1)]
    reasons += _why(list(zip(TRIADS_12II.split("|"), vals)),
                    list(zip(TETRADS_12II.split("|"), qv)))
    if not reasons and not valid12II(*v):
        reasons.append("fails the validity test of check_10_13.valid12II")
    if reasons:
        _emit(sym, 0, note="vanishes: " + "; ".join(reasons), into=into)
        return sp.Integer(0)

    val = sp.simplify(_tw2(*v))
    _emit(sym, val, into=into)
    return val

In [ ]:
# ---- edit and re-run ----  (a2 a3 a4 | b1 b3 b4 | c1 c2 c4 | d1 d2 d3)
J12II = (1, 1, 1,
         1, 1, 1,
         1, 1, 1,
         1, 1, 1)

twelvej2(*J12II);

In [ ]:
if HAS_WIDGETS:
    def _12j2_ui():
        w = [_num(1.0, lo=0.0) for _ in range(12)]
        labs = ["a₂", "a₃", "a₄", "b₁", "b₃", "b₄",
                "c₁", "c₂", "c₄", "d₁", "d₂", "d₃"]
        go  = W.Button(description="Compute", button_style="primary",
                       layout=W.Layout(width="110px"))
        out = W.Output(layout=W.Layout(margin="6px 0 4px 0"))

        def recompute(*_):
            _note(out, "computing &hellip;", colour="#666")
            twelvej2(*[x.value for x in w], into=out)

        for ctl in w:
            ctl.observe(lambda *_: _stale(out), "value")
        go.on_click(recompute)
        _stale(out)
        rows = [_pairs(*[(labs[3 * r + k], w[3 * r + k]) for k in range(3)])
                for r in range(4)]
        return W.VBox([
            _title("$12j$(II) symbol",
                   r"\left\{\begin{array}{cccc} - & a_2 & a_3 & a_4 \\"
                   r" b_1 & - & b_3 & b_4 \\ c_1 & c_2 & - & c_4 \\"
                   r" d_1 & d_2 & d_3 & - \end{array}\right\}"),
            *rows,
            W.HBox([go], layout=W.Layout(margin="6px 0 0 0")),
            out])

    display(_12j2_ui())
else:
    print("ipywidgets not installed - use the edit-and-rerun cell above.")

---
## Cross-check against the algebraic formulas

The algebraic generators in `vmk_algebraic_tables.ipynb` produce a formula in
symbolic $a$ (or $a,b,c$). Substituting numbers into that formula must
reproduce the exact numeric value computed here — two independent routes to the
same quantity, so agreement is a real check on both.

In [ ]:
import gen_8_12_cg_tables as cg8
import gen_9_11_6j_tables as sixj9

def check_cg(a, alpha, b, beta, c):
    """Compare the §8.12 algebraic formula with the exact numeric value."""
    a, alpha, b, beta, c = map(_R, (a, alpha, b, beta, c))
    k = c - a
    formula = cg8.entry(b, beta, k)                       # in terms of c, gamma
    sub = formula.subs({cg8.c: c, cg8.g: alpha + beta})
    exact = sp.simplify(CG(a, alpha, b, beta, c, alpha + beta).doit())
    ok = sp.simplify(sub - exact) == 0
    display(Math(rf"\hspace{{1.5em}}\text{{formula}}\;=\;{sp.latex(sp.simplify(sub))}"
                 rf"\qquad \text{{exact}}\;=\;{root_latex(exact)}"))
    display(HTML(f"<div style='margin-left:{STYLE['indent']};color:"
                 f"{'#2a2' if ok else '#a33'}'>{'agree' if ok else 'DISAGREE'}</div>"))
    return ok


def check_6j(a, b, c, d, e, f):
    """Compare the §9.11 algebraic formula with the exact numeric value."""
    a, b, c, d, e, f = map(_R, (a, b, c, d, e, f))
    m, n = f - b, e - c
    formula = sixj9.entry(d, m, n)
    sub = formula.subs({sixj9.a: a, sixj9.b: b, sixj9.c: c})
    exact = sp.simplify(wigner_6j(a, b, c, d, e, f, prec=None))
    ok = sp.simplify(sub - exact) == 0
    display(Math(rf"\hspace{{1.5em}}\text{{formula}}\;=\;{sp.latex(sp.simplify(sub))}"
                 rf"\qquad \text{{exact}}\;=\;{root_latex(exact)}"))
    display(HTML(f"<div style='margin-left:{STYLE['indent']};color:"
                 f"{'#2a2' if ok else '#a33'}'>{'agree' if ok else 'DISAGREE'}</div>"))
    return ok


check_cg(3, 1, 1, 0, 3)
check_6j(4, 3, 2, 1, 3, 3);

### Notes

* Values are exact throughout: the fraction under the root comes from rational
  arithmetic, and the decimal is derived from it.
* `DIGITS` at the top of the setup cell controls the decimal precision;
  `STYLE` controls the output spacing, as in the algebraic notebook.
* A vanishing coefficient reports *why* it vanishes (which triad failed, or
  which projection is out of range) rather than silently returning zero.
* All three widgets auto-update: numeric evaluation is fast, unlike the
  algebraic $6j$ and $9j$ which need an explicit Compute button.

---
## The book's numerical tables

Reproductions of the tables themselves, with the book's own selection
conditions. Each entry lists the arguments, the exact value as a signed root of
a rational fraction, and the decimal.

| Function | Book table | Selection |
|---|---|---|
| `table_8_11(c)` | 8.11 | $a,b,c\le3$; $a\ge b$, $\alpha\ge0$; $\alpha\ge\beta$ when $a=b$; grouped by $c$ |
| `table_9_9()` | 9.9 | $a,b,d,e$ half-integer, $c,f$ integer; $a\ge b,d,e$, $c\ge f$ |
| `table_9_10()` | 9.10 | $a,b,c$ integer, $d,e,f$ half-integer; $a\ge b\ge c$ |
| `table_9_11()` | 9.11 | all integer; $a\ge b,c,d,e,f$; $b\ge c,e,f$ |
| `table_10_13(c, f, j)` | 10.13–10.14 | $(g,h,j)=(\tfrac12,\tfrac12,0)$ or $(\tfrac12,\tfrac12,1)$; $0\le a\ldots f\le4$, fixed $c,f$ |

Every other coefficient in range reduces to one of these by the symmetry
properties (§8.4, §9.4, §10.4).

In [ ]:
def _halves(lo, hi):
    """lo, lo+1/2, ..., hi."""
    lo, hi = _R(lo), _R(hi)
    return [lo + Rational(i, 2) for i in range(int(2 * (hi - lo)) + 1)]


def _is_int(x):
    return _R(x) % 1 == 0


def show_entries(rows, headers, limit=30, digits=6):
    """Render [(args..., value)] as an array: arguments | exact | decimal."""
    if not rows:
        return HTML("<b>no entries match those conditions</b>")
    shown = rows[:limit]
    ncol = len(headers)
    head = " & ".join([rf"\text{{{h}}}" for h in headers]
                      + [r"\text{exact}", r"\text{decimal}"])
    body = [head + r" \\ \hline"]
    for row in shown:
        *args, v = row
        body.append(" & ".join([_fmt(x) for x in args]
                               + [root_latex(v), sp.latex(decimal(v, digits))]) + r" \\")
    display(Math(r"\begin{array}{" + "c" * ncol + "|c|c}"
                 + " ".join(body) + r"\end{array}"))
    if len(rows) > limit:
        display(HTML(f"<div style='margin-left:{STYLE['indent']};font-size:90%;"
                     f"color:#666'>{len(rows) - limit} further entries not shown "
                     f"&mdash; raise <code>limit</code>.</div>"))


def table_8_11(c, jmax=3, limit=30):
    """Table 8.11: Clebsch-Gordan values for one c.

    Book conditions: a, b, c <= jmax;  a >= b;  alpha >= 0;  and alpha >= beta
    when a = b.  Everything else reduces to these by the symmetries of §8.4.
    """
    c = _R(c)
    rows = []
    for a in _halves(0, jmax):
        for b in _halves(0, a):                      # a >= b
            if not _tri(a, b, c):
                continue
            for al in _halves(0, a):                 # alpha >= 0
                for be in _halves(-b, b):
                    if a == b and al < be:           # alpha >= beta when a = b
                        continue
                    ga = al + be
                    if abs(ga) > c or (a - al) % 1 or (b - be) % 1:
                        continue
                    v = sp.simplify(CG(a, al, b, be, c, ga).doit())
                    if v != 0:
                        rows.append((a, al, b, be, ga, v))
    show_entries(rows, ["a", "α", "b", "β", "γ"], limit=limit)
    return rows


table_8_11(c=2, jmax=2);

In [ ]:
def _sixj_rows(jmax, keep):
    rows = []
    vals = _halves(Rational(1, 2), jmax)             # all arguments differ from zero
    for a in vals:
        for b in vals:
            for c in vals:
                if not _tri(a, b, c):
                    continue
                for d in vals:
                    for e in vals:
                        for f in vals:
                            if not keep(a, b, c, d, e, f):
                                continue
                            if not (_tri(a, e, f) and _tri(d, b, f) and _tri(d, e, c)):
                                continue
                            v = sp.simplify(wigner_6j(a, b, c, d, e, f, prec=None))
                            if v != 0:
                                rows.append((a, b, c, d, e, f, v))
    return rows


def table_9_9(jmax=3, limit=30):
    """Table 9.9: a,b,d,e half-integer, c,f integer; a>=b,d,e and c>=f;
    if a=b then d>=e; if c=f then b>=e."""
    def keep(a, b, c, d, e, f):
        return (not _is_int(a) and not _is_int(b) and not _is_int(d) and not _is_int(e)
                and _is_int(c) and _is_int(f)
                and a >= b and a >= d and a >= e and c >= f
                and (d >= e if a == b else True) and (b >= e if c == f else True))
    rows = _sixj_rows(jmax, keep)
    show_entries(rows, ["a", "b", "c", "d", "e", "f"], limit=limit)
    return rows


def table_9_10(jmax=3, limit=30):
    """Table 9.10: a,b,c integer, d,e,f half-integer; a>=b>=c;
    if a=b then d>=e; if b=c then e>=f."""
    def keep(a, b, c, d, e, f):
        return (_is_int(a) and _is_int(b) and _is_int(c)
                and not _is_int(d) and not _is_int(e) and not _is_int(f)
                and a >= b >= c
                and (d >= e if a == b else True) and (e >= f if b == c else True))
    rows = _sixj_rows(jmax, keep)
    show_entries(rows, ["a", "b", "c", "d", "e", "f"], limit=limit)
    return rows


def table_9_11(jmax=3, limit=30):
    """Table 9.11: all integer; a>=b,c,d,e,f; b>=c,e,f;
    if a=b then d>=e; if b=c then e>=f; if a=d then c>=f."""
    def keep(a, b, c, d, e, f):
        return (all(_is_int(x) for x in (a, b, c, d, e, f))
                and a >= max(b, c, d, e, f) and b >= max(c, e, f)
                and (d >= e if a == b else True)
                and (e >= f if b == c else True)
                and (c >= f if a == d else True))
    rows = _sixj_rows(jmax, keep)
    show_entries(rows, ["a", "b", "c", "d", "e", "f"], limit=limit)
    return rows


table_9_11(jmax=2);

In [ ]:
def table_10_13(c, f, j, jmax=4, limit=30):
    """Tables 10.13-10.14: 9j values with (g,h,j) = (1/2, 1/2, j), j = 0 or 1,
    at fixed c and f, with 0 <= a,b,d,e <= jmax."""
    c, f, j = _R(c), _R(f), _R(j)
    g = h = Rational(1, 2)
    if j not in (0, 1):
        return HTML("<b>the book's tables have j = 0 or 1 only</b>")
    rows = []
    for a in _halves(0, jmax):
        for b in _halves(0, jmax):
            if not _tri(a, b, c):
                continue
            for d in _halves(0, jmax):
                if not _tri(a, d, g):
                    continue
                for e in _halves(0, jmax):
                    if not (_tri(d, e, f) and _tri(b, e, h) and _tri(c, f, j)):
                        continue
                    v = sp.simplify(wigner_9j(a, b, c, d, e, f, g, h, j, prec=None))
                    if v != 0:
                        rows.append((a, b, d, e, v))
    display(Math(r"\hspace{1.5em}\begin{Bmatrix} a & b & " + _fmt(c)
                 + r" \\ d & e & " + _fmt(f) + r" \\ \tfrac12 & \tfrac12 & "
                 + _fmt(j) + r"\end{Bmatrix}"))
    show_entries(rows, ["a", "b", "d", "e"], limit=limit)
    return rows


table_10_13(c=1, f=1, j=0, jmax=2);

Each generator returns the rows as well as displaying them, so they can be
fed onward — compared against the OCR'd tables in the `.tex`, for instance, or
widened with `jmax` beyond the range the book prints.

`limit` only caps what is *shown*; the returned list is complete.